# Drive Shaping Methods

A quantum program in the Rydberg analog model is defined as a time-dependent drive Hamiltonian that is imposed on the qubits (in addition to the interaction Hamiltonian). The drive shaping configuration part (the `drive_shaping` field of `SolverConfig`) defines how the drive parameters are constructed.

In this notebook, we show how to use different drive shaping methods:

- `ADIABATIC` *(Has no parameters to be customized).*
- `OPTIMIZED` *(Has seven parameters that can be customized).*

We choose the method when defining the `SolverConfig` configuration, with `drive_shaping_method = DriveType.(method)`

**Default SolverConfig parameters to run a quantum approach:**
- use_quantum: bool = False (*for using the drive shaping methods, we have to set it to `True`.*)
- backend: LocalEmulator | RemoteEmulator | QPU (by default, use a LocalEmulator based on qutip)
- device: Device = DigitalAnalogDevice() (also available: `AnalogDevice()`)
- embedding_method: str | EmbedderType | None = EmbedderType.GREEDY (also available: `BLADE`)
- drive_shaping_method: str | DriveType | None = DriveType.ADIABATIC (also available: `OPTIMIZED`)


In [ ]:
import torch

from qubosolver.qubo_instance import QUBOInstance
from qubosolver.config import SolverConfig, DriveShapingConfig
from qubosolver.qubo_types import DriveType
from qubosolver.solver import QuboSolver

import matplotlib.pyplot as plt
plt.rcParams["animation.html"] = "jshtml"

%matplotlib inline

** Create the <code>QUBOInstance</code> **

Here, we have a 3x3 QUBO matrix with negative diagonal and positive off-diagonal terms.

In [ ]:
coefficients = torch.tensor([[-1.0, 0.5, 0.2], [0.5, -2.0, 0.3], [0.2, 0.3, -3.0]])
instance = QUBOInstance(coefficients)

## Standard Adiabatic

Default method


In [ ]:
default_config = SolverConfig.from_kwargs(
    use_quantum=True, drive_shaping=DriveShapingConfig(drive_shaping_method=DriveType.ADIABATIC),
)
solver = QuboSolver(instance, default_config)

solution = solver.solve()
print(solution)

If needed, one can access the drive used and draw the sequence used by the quantum solver as follows:  

In [ ]:
embedding = solver.embedding()
drive = solver.drive(embedding)[0]
solver.draw_sequence(drive, embedding)

## Optimized Drive shaping 

#### Parameters to customize:

For the OPTIMIZED drive shaping, we have the following parameters:
- optimized_n_calls: Number of calls for the optimization process.
- optimized_re_execute_opt_drive: Whether to re-run the optimal drive sequence after optimization.
- optimized_initial_omega_parameters: Default initial omega parameters for the drive. Defaults to (5, 10, 5).
- optimized_initial_detuning_parameters: Default initial detuning parameters for the drive. Defaults to (-10, 0, 10).

#### Default configuration

In [ ]:
default_config = SolverConfig.from_kwargs(
    use_quantum=True, drive_shaping=DriveShapingConfig(drive_shaping_method=DriveType.OPTIMIZED),
)
solver = QuboSolver(instance, default_config)

solution = solver.solve()
print(solution)

#### Changing <code>optimized_n_calls</code>

In [ ]:
default_config = SolverConfig.from_kwargs(
    use_quantum=True, drive_shaping=DriveShapingConfig(drive_shaping_method=DriveType.OPTIMIZED, optimized_n_calls=13),
)
solver = QuboSolver(instance, default_config)

solution = solver.solve()
print(solution)

#### Changing the initial parameters

Initial parameters of the optimization procedure can be changed as `optimized_initial_omega_parameters` and `optimized_initial_detuning_parameters`

In [ ]:
default_config = SolverConfig.from_kwargs(
    use_quantum=True, drive_shaping=DriveShapingConfig(drive_shaping_method=DriveType.OPTIMIZED, optimized_initial_omega_parameters=[2.0, 15.0, 5.0,], optimized_initial_detuning_parameters=[-45.0, 0.0, 25.0]),
)
solver = QuboSolver(instance, default_config)

solution = solver.solve()
print(solution)

#### Changing <code>optimized_re_execute_opt_drive</code>

In [ ]:
default_config = SolverConfig.from_kwargs(
    use_quantum=True, drive_shaping=DriveShapingConfig(drive_shaping_method=DriveType.OPTIMIZED, optimized_re_execute_opt_drive=True),
)
solver = QuboSolver(instance, default_config)

solution = solver.solve()
print(solution)

#### Adding custom functions

One can change the drive shaping method by incorporating custom functions for:
- Evaluating a candidate bitstring and QUBO via `optimized_custom_qubo_cost`
- Performing optimization with a different objective than the best cost via `optimized_custom_objective`
- Adding callback functions via `optimized_callback_objective`.

In [ ]:
from qubosolver.utils.qubo_eval import calculate_qubo_cost

# example of penalization
def penalized_qubo(bitstring: str, QUBO: torch.Tensor) -> float:
    return calculate_qubo_cost(bitstring, QUBO) + 2 * bitstring.count("0")

# example of saving intermediate results
opt_results = list()
def callback(d: dict) -> None:
    opt_results.append(d)

# example of using an average cost
def average_ojective(
    bitstrings: list,
    counts: list,
    probabilities: list,
    costs: list,
    best_cost: float,
    best_bitstring: str,
) -> float:
    return sum([p * c for p, c in zip(probabilities, costs)])

drive_shaping=DriveShapingConfig(drive_shaping_method=DriveType.OPTIMIZED, 
    optimized_re_execute_opt_drive=True,
    optimized_custom_qubo_cost=penalized_qubo, 
    optimized_callback_objective=callback,
    optimized_custom_objective = average_ojective,
)

config = SolverConfig(
    use_quantum=True, 
    drive_shaping=drive_shaping,
)

In [ ]:
solver = QuboSolver(instance, config)
solution = solver.solve()
len(opt_results), opt_results[-1]

In [ ]:
solution